In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Configure options for headless browsing
opts = Options()
opts.add_argument("--headless=new")
opts.add_argument("--window-size=1920,1080")

# Initialize the driver
driver = webdriver.Chrome(options=opts)

try:
    # Navigate to the target page
    driver.get("https://news.ycombinator.com/") # Example URL

    # Wait for content to ensure the page is loaded
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "athing"))
    )

    # Get the full page source after JavaScript execution
    full_html_source = driver.page_source

    print("Scraped page source snippet (first 500 characters):")
    print(full_html_source[:500])


    # Extract specific data (e.g., article titles)
    stories = driver.find_elements(By.CLASS_NAME, "athing")
    print(f"\nFound {len(stories)} stories.")
    for story in stories[:3]:
        title_link = story.find_element(By.CSS_SELECTOR, "span.titleline > a")
        print(f"- {title_link.text}: {title_link.get_attribute('href')}")
        
    # 2. Scraping CSS File Links
    # CSS is usually found in <link> tags with rel="stylesheet"
    css_links = driver.find_elements(By.TAG_NAME, "link")
    print("--- CSS FILES ---")
    for link in css_links:
        rel = link.get_attribute("rel")
        if rel == "stylesheet":
            href = link.get_attribute("href")
            print(f"Found CSS: {href}")

    # 3. Scraping JavaScript File Links
    # JS is found in <script> tags with a "src" attribute
    js_scripts = driver.find_elements(By.TAG_NAME, "script")
    print("\n--- JAVASCRIPT FILES ---")
    for script in js_scripts:
        src = script.get_attribute("src")
        if src:  # Only print if it's an external file, not inline code
            print(f"Found JS: {src}")


finally:
    # Always close the browser
    driver.quit()


In [ ]:
import os
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
import jsbeautifier

# Setup folders
BASE_DIR = "Dataset/website_dataset"
os.makedirs(f"{BASE_DIR}/css", exist_ok=True)
os.makedirs(f"{BASE_DIR}/js", exist_ok=True)

opts = Options()
opts.add_argument("--headless=new")
driver = webdriver.Chrome(options=opts)

def download_file(url, folder):
    if not url or not url.startswith("http"):
        return
    
    # Create a filename from the end of the URL
    filename = url.split("/")[-1].split("?")[0] # Remove version queries like ?v=1.2
    if not filename:
        return

    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            with open(f"{BASE_DIR}/{folder}/{filename}", "w", encoding="utf-8") as f:
                f.write(response.text)
            print(f"Successfully saved: {filename}")
    except Exception as e:
        print(f"Failed to download {url}: {e}")

try:
    driver.get("https://google.com/") # Replace with your target
    WebDriverWait(driver, 15).until(EC.visibility_of_element_located((By.TAG_NAME, "body")))
    
    final_html = driver.execute_script("return document.body.innerHTML")
    with open("structure.html", "w", encoding="utf-8") as f:
      f.write(final_html)

    # 1. Download CSS
    links = driver.find_elements(By.TAG_NAME, "link")
    for link in links:
        if link.get_attribute("rel") == "stylesheet":
            download_file(link.get_attribute("href"), "css")

    # 2. Download JS
    scripts = driver.find_elements(By.TAG_NAME, "script")
    for script in scripts:
        src = script.get_attribute("src")
        src = jsbeautifier.beautify(src)
        if src:
            download_file(src, "js")

finally:
    driver.quit()
    print("\nDownload complete. Check the 'website_dataset' folder.")

Failed to download https://gc.kis.v2.scr.kaspersky-labs.com/E3E8934C-235A-4B0E-825A-35A08381A191/abn/main.css?attr=aHR0cHM6Ly93d3cuZ29vZ2xlLmNvbS8: HTTPSConnectionPool(host='gc.kis.v2.scr.kaspersky-labs.com', port=443): Max retries exceeded with url: /E3E8934C-235A-4B0E-825A-35A08381A191/abn/main.css?attr=aHR0cHM6Ly93d3cuZ29vZ2xlLmNvbS8 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1077)')))
Successfully saved: m=cdos,cr,hsm,jsa,mb4ZUb,cEt90b,SNUn3,qddgKe,sTsDMc,dtl0hd,eHDfl,YV5bee,d,csi
Successfully saved: rs=AA2YrTuB9pNk5GDNCjpS0-TPiTc9nq0jFg
Successfully saved: m=cdos,cr,hsm,jsa,mb4ZUb,cEt90b,SNUn3,qddgKe,sTsDMc,dtl0hd,eHDfl,YV5bee,d,csi
Successfully saved: rs=AA2YrTtjvfjmkSNz5TMGsTjd3FbbhfTWYw
Failed to download https://www.google.com/xjs/_/js/k=xjs.hd.en_GB.UTKhnQM0sLE.2019.O/ck=xjs.hd.J-ap2_wOjJs.L.W.O/am=AEAQAQAAAAAAAAAIAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAEAAAAgAAK